<a href="https://colab.research.google.com/github/yasyamauchi/education/blob/main/BME_AI_outliers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 人工知能 補助教材 (外れ値の検定)  
### 東洋大学 生命科学部/理工学部 生体医工学科

更新履歴：  
2026/8/17  
* 初版  

# 1)第1と第3四分位数の区間から外れるものを外れ値とする考え方  

四分位数の意味は理解しているものと仮定する．  
60個のランダムに生成されたデータを「箱ひげ図」(boxplot)に表示する．青線は第2四分位数，すなわち中央値である．  
赤丸が外れ値となる．この考え方では，かなりのデータ(半分ぐらい？)が外れ値になってしまうことがわかる．

In [ ]:
# @title
# 1. 必要なライブラリのインストールとインポート
!pip install japanize-matplotlib -q

import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np
import pandas as pd

# 再現性のための乱数シード
np.random.seed(42)

# 2. サンプルデータの生成（平均50、標準偏差10の正規分布データ 60件）
data = np.random.normal(loc=50, scale=10, size=60)

# 3. 第1四分位数 (Q1) と 第3四分位数 (Q3) の算出
q1 = np.percentile(data, 25)
q3 = np.percentile(data, 75)

# 条件に基づく外れ値の抽出（Q1未満 または Q3超過）
outliers = data[(data < q1) | (data > q3)]
normal_data = data[(data >= q1) & (data <= q3)]

# 4. 箱ひげ図の描画
fig, ax = plt.subplots(figsize=(8, 6))

# whis=0 に設定することで、ひげを伸ばさず Q1〜Q3 の外側すべてを外れ値マーカーとして描画
bp = ax.boxplot(
    data,
    whis=0,
    patch_artist=True,
    boxprops=dict(facecolor="skyblue", color="steelblue", alpha=0.7),
    medianprops=dict(color="darkblue", linewidth=2),
    flierprops=dict(
        marker="o",
        markerfacecolor="red",
        markeredgecolor="darkred",
        markersize=6,
        alpha=0.6,
    ),
)

# 四分位数の基準線を追加
ax.axhline(
    q3, color="darkorange", linestyle="--", label=f"第3四分位数 (Q3): {q3:.2f}"
)
ax.axhline(
    q1, color="forestgreen", linestyle="--", label=f"第1四分位数 (Q1): {q1:.2f}"
)

# グラフ装飾
ax.set_title("箱ひげ図（$Q_1$ 〜 $Q_3$ の区間外を外れ値とするデモ）", fontsize=14)
ax.set_ylabel("値", fontsize=12)
ax.set_xticklabels(["データセット"])
ax.legend(loc="upper right")
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.show()

# 5. 数値データの確認
print(f"総データ数: {len(data)}")
print(f"第1四分位数 (Q1): {q1:.2f}")
print(f"第3四分位数 (Q3): {q3:.2f}")
print(f"外れ値の件数: {len(outliers)} 件")
print(f"外れ値一覧:\n{np.sort(outliers)}")

# 2)平均からの乖離が2σ以上であるものを外れ値とする考え方  

σ(標準偏差)の意味は理解しているものと仮定する．  
同じく，60個のランダムに生成されたデータである．真ん中の青い線は通常「エラーバー」とよばれる，平均値±σを表す．赤い線は±２σを表し，そこを外れたデータを外れ値とする．  
この考え方では外れ値は非常に少ないはず．

In [ ]:
# @title
# 1. 必要なライブラリのインストールとインポート
!pip install japanize-matplotlib -q

import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np

# 2. サンプルデータの生成（前回と同じシード・パラメータ）
np.random.seed(42)
data = np.random.normal(loc=50, scale=10, size=60)

# 3. 統計量の計算
mean = np.mean(data)
std = np.std(data, ddof=1)  # 不偏標準偏差

lower_bound_1s = mean - 1 * std
upper_bound_1s = mean + 1 * std
lower_bound_2s = mean - 2 * std
upper_bound_2s = mean + 2 * std

# 外れ値の判定 (|データ - 平均| >= 2σ)
is_outlier = (data < lower_bound_2s) | (data > upper_bound_2s)
indices = np.arange(len(data))

# 4. グラフの描画
fig, ax = plt.subplots(figsize=(11, 6))

# --- 各データ点の散布図 ---
ax.scatter(
    indices[~is_outlier],
    data[~is_outlier],
    color="steelblue",
    alpha=0.75,
    label="正常値 ($\\mu \\pm 2\\sigma$ 以内)",
    zorder=3,
)
ax.scatter(
    indices[is_outlier],
    data[is_outlier],
    color="red",
    s=80,
    edgecolors="darkred",
    label="外れ値 ($|x - \\mu| \\geq 2\\sigma$)",
    zorder=4,
)

# --- 標準偏差エラーバーの重ね合わせ（中央位置 x=29.5 に配置） ---
center_x = (len(data) - 1) / 2

# 2σ エラーバー（外れ値の境界）
ax.errorbar(
    center_x,
    mean,
    yerr=2 * std,
    fmt="none",
    ecolor="crimson",
    elinewidth=2.5,
    capsize=12,
    capthick=2.5,
    label=f"外れ値判定境界 ($\\mu \\pm 2\\sigma$: {lower_bound_2s:.2f} 〜 {upper_bound_2s:.2f})",
    zorder=5,
)

# 1σ エラーバー（標準偏差 1σ）
ax.errorbar(
    center_x,
    mean,
    yerr=1 * std,
    fmt="D",
    color="darkblue",
    ecolor="darkblue",
    elinewidth=4.5,
    capsize=8,
    capthick=3,
    markersize=8,
    label=f"平均値 & 標準偏差 ($\\mu \\pm 1\\sigma$: {mean:.2f} $\\pm$ {std:.2f})",
    zorder=6,
)

# --- 補助線と帯 ---
ax.axhline(mean, color="darkblue", linestyle="-", linewidth=1.2, alpha=0.6)
ax.axhline(
    upper_bound_2s,
    color="darkorange",
    linestyle="--",
    linewidth=1.2,
    alpha=0.6,
)
ax.axhline(
    lower_bound_2s,
    color="forestgreen",
    linestyle="--",
    linewidth=1.2,
    alpha=0.6,
)
ax.axhspan(
    lower_bound_2s,
    upper_bound_2s,
    color="skyblue",
    alpha=0.15,
    label="正常範囲 ($\\mu \\pm 2\\sigma$)",
)

# 装飾
ax.set_title(
    "各データ点の散布図と標準偏差エラーバー（$1\\sigma$ / $2\\sigma$）の重ね合わせ",
    fontsize=14,
)
ax.set_xlabel("データインデックス (0〜59)", fontsize=11)
ax.set_ylabel("値", fontsize=11)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

# 5. 数値の確認
print(f"総データ数: {len(data)}")
print(f"平均値 (μ): {mean:.2f}")
print(f"標準偏差 (1σ): {std:.2f}")
print(f"1σ範囲 (μ ± 1σ): {lower_bound_1s:.2f} 〜 {upper_bound_1s:.2f}")
print(f"2σ範囲 (μ ± 2σ): {lower_bound_2s:.2f} 〜 {upper_bound_2s:.2f}")
print(f"外れ値の件数: {np.sum(is_outlier)} 件")